In [1]:
import polars as pl
import requests
from bs4 import BeautifulSoup
import requests
import os
from datetime import datetime

In [98]:
# Define the parameters
params = {
    "locationIdentifier": "REGION^87490",
    "maxBedrooms": 3,
    "minBedrooms": 2,
    "maxPrice": 450000,
    "radius": 15.0,
    "index": 0,
    "propertyTypes": "bungalow,detached,semi-detached,terraced",
    "includeSSTC": "false",
    "mustHave": "",
    "dontShow": "newHome",
    "furnishTypes": "",
    "keywords": ""
}

# Construct the URL using f-string
url = (
    f"https://www.rightmove.co.uk/property-for-sale/find.html?"
    f"locationIdentifier={params['locationIdentifier']}&"
    f"maxBedrooms={params['maxBedrooms']}&"
    f"minBedrooms={params['minBedrooms']}&"
    f"maxPrice={params['maxPrice']}&"
    f"radius={params['radius']}&"
    f"index={params['index']}&"
    f"propertyTypes={params['propertyTypes']}&"
    f"includeSSTC={params['includeSSTC']}&"
    f"mustHave={params['mustHave']}&"
    f"dontShow={params['dontShow']}&"
    f"furnishTypes={params['furnishTypes']}&"
    f"keywords={params['keywords']}"
)

print(url)



https://www.rightmove.co.uk/property-for-sale/find.html?locationIdentifier=REGION^87490&maxBedrooms=3&minBedrooms=2&maxPrice=450000&radius=15.0&index=0&propertyTypes=bungalow,detached,semi-detached,terraced&includeSSTC=false&mustHave=&dontShow=newHome&furnishTypes=&keywords=


In [99]:
# Make a request to the URL
r = requests.get(url)

# Check the status code
if r.status_code == 200:
    print(f"Success Accessing page, status code: {r.status_code}")
else:
    raise Exception(f"Failed to access page, status code {r.status_code}")

Success Accessing page, status code: 200


In [100]:
# Parse the HTML content
soup = BeautifulSoup(r.text, 'html.parser')

In [102]:
pl.DataFrame({'col1' : [1,2,3], 'col2' : 'a b c'.split()})

col1,col2
i64,str
1,"""a"""
2,"""b"""
3,"""c"""


In [26]:
df = pl.read_parquet(r'../data/raw/scraped_html')

In [36]:
df = pl.read_parquet(r'../data/raw/scraped_html')
latest_run_id = df.sort(by=['run_id'], descending=True).limit(1).select('run_id').to_series().item()


7

In [95]:
properties = soup.find_all('div', {'class': 'l-searchResult is-list'})

In [97]:
properties

[<div class="l-searchResult is-list" data-bind="attr: { 'id': 'property-' + id() }, css: { 'is-hidden': hasNoDetails, 'is-list': isList}" data-test="propertyCard-0" id="property-150666707">
 <div class="propertyCard propertyCard--premium propertyCard--featured" data-bind="css: {'propertyCard--premium': premiumListing, 'propertyCard-hideMobileIconsMVT': hideMobileIcons, 'propertyCard--featured': featuredProperty, 'propertyCard--saved': isPropertySaved }" itemscope="" itemtype="http://schema.org/Residence">
 <a class="propertyCard-anchor" id="prop150666707"></a>
 <div class="propertyCard-wrapper">
 <div class="propertyCard-images">
 <div class="propertyCard-main-img-mask aspect-3x2">
 <div class="propertyCard-main-img">
 <a class="propertyCard-img-link aspect-3x2" data-test="property-img" href="/properties/150666707#/?channel=RES_BUY">
 <div class="propertyCard-img">
 <span class="no-svg-camera camera">
 <svg>
 <use xlink:href="#core-icon--camera"></use>
 </svg>
 </span>
 <img alt="Prope

In [96]:
for property in properties[0]:
        
    # Extract property price
    price_tag = property.find('div', {'class' : 'propertyCard-priceValue'})
    price = price_tag.get_text(strip=True) if price_tag else 'Not Available'

    # Extract property address
    address_tag = property.find('address', class_='propertyCard-address')
    address = address_tag.get_text(strip=True) if address_tag else 'Not Available'

    # Extract property description
    description_tag = property.find('span', itemprop='description')
    description = description_tag.get_text(strip=True) if description_tag else 'Not Available'

    # Extract number of bedrooms
    bedrooms_tag = property.find('span', class_='no-svg-bed-icon')
    bedrooms = bedrooms_tag.find_next('span').get_text(strip=True) if bedrooms_tag else 'Not Available'

    # Extract number of bathrooms
    bathrooms_tag = property.find('span', class_='no-svg-bathroom-icon')
    bathrooms = bathrooms_tag.find_next('span').get_text(strip=True) if bathrooms_tag else 'Not Available'

    # Extract property type (e.g., "Terraced")
    property_type_tag = property.find('span', class_='text')
    property_type = property_type_tag.get_text(strip=True) if property_type_tag else 'Not Available'

    # Extract agent name and phone number
    agent_name_tag = property.find('div', class_='propertyCard-branchLogo-link')
    agent_name = agent_name_tag['title'] if agent_name_tag else 'Not Available'

    phone_tag = property.find('a', class_='propertyCard-contactsPhoneNumber')
    phone_number = phone_tag.get_text(strip=True) if phone_tag else 'Not Available'


    # Print the extracted data
    print(f"Price: {price}")
    print(f"Address: {address}")
    print(f"Description: {description}")
    print(f"Bedrooms: {bedrooms}")
    print(f"Bathrooms: {bathrooms}")
    print(f"Property Type: {property_type}")
    print(f"Agent: {agent_name}")
    print(f"Phone: {phone_number}")


TypeError: slice indices must be integers or None or have an __index__ method

In [86]:
properties[0].find('div', class_='propertyCard-priceValue')

<div class="propertyCard-priceValue" data-bind="text: price.displayPrices()[0] ? price.displayPrices()[0].displayPrice : ''">£500,000        </div>

In [81]:
print('https://www.rightmove.co.uk/')
properties[0].find("a", class_="propertyCard-link")['href']

https://www.rightmove.co.uk/


'/properties/157326728#/?channel=RES_BUY'